In [3]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from pprint import pprint

## Task 1: Replace Hardcoded Prompts

Convert the hardcoded prompt into a reusable LangChain PromptTemplate.

In [4]:
# Reusable template for explaining any topic.
task1_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms for beginners."
)


def explain_topic(topic):
    """Return a beginner-friendly explanation prompt for the given topic."""
    return task1_template.format(topic=topic)


print(explain_topic("Machine Learning"))

Explain Machine Learning in simple terms for beginners.


## Task 2: Multi-Input Prompt System

Build a prompt template with multiple inputs: topic, audience, and tone.

In [5]:
# Multi-input prompt template.
task2_template = PromptTemplate(
    input_variables=["topic", "audience", "tone"],
    template="Explain {topic} for {audience} in a {tone} tone."
)


def build_multi_input_prompt(topic, audience, tone):
    """Create a prompt using topic, audience, and tone."""
    return task2_template.format(topic=topic, audience=audience, tone=tone)


task2_test_cases = [
    {"topic": "AI", "audience": "beginners", "tone": "friendly"},
    {"topic": "Python", "audience": "kids", "tone": "fun"},
    {"topic": "Deep Learning", "audience": "engineers", "tone": "technical"}
]

for case in task2_test_cases:
    print(build_multi_input_prompt(case["topic"], case["audience"], case["tone"]))

Explain AI for beginners in a friendly tone.
Explain Python for kids in a fun tone.
Explain Deep Learning for engineers in a technical tone.


## Task 3: Prompt Variations Engine

Create three different prompt templates for teaching, interview, and storytelling styles.

In [6]:
# Separate templates for each prompt style.
teaching_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} clearly step by step."
)

interview_template = PromptTemplate(
    input_variables=["topic"],
    template="Ask 3 interview questions about {topic}."
)

storytelling_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} as a story."
)

STYLE_TEMPLATES = {
    "teaching": teaching_template,
    "interview": interview_template,
    "storytelling": storytelling_template
}


def generate_prompt_variations(topic):
    """Return prompt variations for all supported styles."""
    variations = {}
    for style_name, template in STYLE_TEMPLATES.items():
        variations[style_name] = template.format(topic=topic)
    return variations


task3_output = generate_prompt_variations("Machine Learning")
pprint(task3_output)

{'interview': 'Ask 3 interview questions about Machine Learning.',
 'storytelling': 'Explain Machine Learning as a story.',
 'teaching': 'Explain Machine Learning clearly step by step.'}


## Task 4: ChatPromptTemplate System

Build a role-based chat prompt system using system and user messages.

In [7]:
# Role instructions are kept separate from the chat template so the design stays modular.
ROLE_BEHAVIOR = {
    "teacher": "You are a patient teacher who explains concepts clearly, simply, and step by step.",
    "interviewer": "You are an interviewer who asks thoughtful and relevant questions about the given topic.",
    "motivator": "You are a motivator who explains concepts in an encouraging and inspiring way."
}

chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "{system_instruction}"),
        ("human", "Topic: {topic}")
    ]
)


def build_chat_prompt(role, topic):
    """Return role-based chat messages for a given topic."""
    normalized_role = role.strip().lower()

    if normalized_role not in ROLE_BEHAVIOR:
        raise ValueError(
            "Invalid role. Allowed roles are: teacher, interviewer, motivator."
        )

    prompt_value = chat_template.format_prompt(
        system_instruction=ROLE_BEHAVIOR[normalized_role],
        topic=topic
    )
    return prompt_value.to_messages()


task4_messages = build_chat_prompt("teacher", "Neural Networks")
for message in task4_messages:
    print(type(message).__name__ + ":", message.content)


SystemMessage: You are a patient teacher who explains concepts clearly, simply, and step by step.
HumanMessage: Topic: Neural Networks


## Task 5: Input Validation Layer

Create a validation function for audience and tone.

In [8]:
ALLOWED_AUDIENCES = {"beginner", "intermediate", "expert"}
ALLOWED_TONES = {"formal", "casual", "fun"}

DEFAULT_AUDIENCE = "beginner"
DEFAULT_TONE = "formal"


def validate_inputs(audience, tone, strict=False):
    """
    Validate audience and tone.

    If strict is True, invalid values raise a ValueError.
    If strict is False, invalid values are replaced with defaults.
    """
    normalized_audience = audience.strip().lower()
    normalized_tone = tone.strip().lower()

    invalid_fields = []

    if normalized_audience not in ALLOWED_AUDIENCES:
        invalid_fields.append("audience")

    if normalized_tone not in ALLOWED_TONES:
        invalid_fields.append("tone")

    if invalid_fields and strict:
        raise ValueError(
            "Invalid input for: " + ", ".join(invalid_fields)
            + ". Allowed audience values: beginner, intermediate, expert."
            + " Allowed tone values: formal, casual, fun."
        )

    if normalized_audience not in ALLOWED_AUDIENCES:
        normalized_audience = DEFAULT_AUDIENCE

    if normalized_tone not in ALLOWED_TONES:
        normalized_tone = DEFAULT_TONE

    return normalized_audience, normalized_tone


print(validate_inputs("beginner", "fun"))
print(validate_inputs("student", "playful"))


('beginner', 'fun')
('beginner', 'formal')


## Task 6: Prompt Generator App

Create a function generate_prompt(topic, audience, tone, style) using PromptTemplate.

In [9]:
# Style-specific logic is stored separately so one reusable template can be used
# with many different inputs.
STYLE_INSTRUCTIONS = {
    "teaching": "Explain the topic clearly step by step with helpful learning points.",
    "interview": "Create 3 interview-style questions that test understanding of the topic.",
    "storytelling": "Explain the topic as an engaging story with a clear and memorable flow."
}

generator_template = PromptTemplate(
    input_variables=["topic", "audience", "tone", "style", "style_instruction"],
    template=(
        "Topic: {topic}\n"
        "Audience: {audience}\n"
        "Tone: {tone}\n"
        "Style: {style}\n"
        "Instruction: {style_instruction}"
    )
)


def generate_prompt(topic, audience, tone, style, strict=False):
    """
    Generate a structured prompt after validating input values.

    Supported styles:
    - teaching
    - interview
    - storytelling
    """
    validated_audience, validated_tone = validate_inputs(audience, tone, strict=strict)
    normalized_style = style.strip().lower()

    if normalized_style not in STYLE_INSTRUCTIONS:
        raise ValueError(
            "Invalid style. Allowed styles are: teaching, interview, storytelling."
        )

    return generator_template.format(
        topic=topic.strip(),
        audience=validated_audience,
        tone=validated_tone,
        style=normalized_style,
        style_instruction=STYLE_INSTRUCTIONS[normalized_style]
    )


print(generate_prompt("Neural Networks", "beginner", "fun", "storytelling"))

Topic: Neural Networks
Audience: beginner
Tone: fun
Style: storytelling
Instruction: Explain the topic as an engaging story with a clear and memorable flow.


## Task 7: Template Reusability Test

Use one reusable template with five different inputs to show the same structure producing different outputs.


In [10]:
reusability_test_cases = [
    {"topic": "Machine Learning", "audience": "beginner", "tone": "formal", "style": "teaching"},
    {"topic": "Python", "audience": "intermediate", "tone": "casual", "style": "storytelling"},
    {"topic": "Deep Learning", "audience": "expert", "tone": "formal", "style": "interview"},
    {"topic": "Data Science", "audience": "beginner", "tone": "fun", "style": "teaching"},
    {"topic": "Natural Language Processing", "audience": "intermediate", "tone": "casual", "style": "storytelling"}
]

for index, case in enumerate(reusability_test_cases, start=1):
    print("-" * 80)
    print("Test Case " + str(index))
    print(
        generate_prompt(
            topic=case["topic"],
            audience=case["audience"],
            tone=case["tone"],
            style=case["style"]
        )
    )


--------------------------------------------------------------------------------
Test Case 1
Topic: Machine Learning
Audience: beginner
Tone: formal
Style: teaching
Instruction: Explain the topic clearly step by step with helpful learning points.
--------------------------------------------------------------------------------
Test Case 2
Topic: Python
Audience: intermediate
Tone: casual
Style: storytelling
Instruction: Explain the topic as an engaging story with a clear and memorable flow.
--------------------------------------------------------------------------------
Test Case 3
Topic: Deep Learning
Audience: expert
Tone: formal
Style: interview
Instruction: Create 3 interview-style questions that test understanding of the topic.
--------------------------------------------------------------------------------
Test Case 4
Topic: Data Science
Audience: beginner
Tone: fun
Style: teaching
Instruction: Explain the topic clearly step by step with helpful learning points.
------------------